In [ ]:
## Baseline detection with PSI derivatives

# PSI Baseline Estimation and Time-in-Range Analysis

#This notebook contains the signal processing pipeline used in:

#Re-evaluating Performance Metrics for Closed-Loop Total Intravenous Anesthesia Systems: Insights from a Post-Market Evaluation

#Functions included:
#- PSI smoothing
#- Maintenance phase detection
#- Baseline estimation
#- Time-in-range analysis

#Author: Laura Gomez et al.

# ============================================================
# 1. IMPORT LIBRARIES AND LOAD THE DATA FILE
# ============================================================

# Import operating system utilities to inspect available files.
import os

# Import numerical, tabular, and plotting libraries.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Optional Google Colab upload step.
# If this script is running in Google Colab, this block allows the user to upload the data file.
# If this script is running locally, the block is skipped and the file must already be in the working folder.
try:
    from google.colab import files
    uploaded = files.upload()
except Exception:
    uploaded = None

# List files in the current working directory so the user can confirm the uploaded file name.
print('Files in current directory:')
print(os.listdir())

# Original file name from the notebook.
# Change this only if your uploaded data file has a different name.
!pip install xlrd
fname = "311202233204\xa0_1070810290.xls"

# Read the file as a tab-separated table.
# Even though the extension is .xls, the original notebook reads it with read_csv using a tab separator.
df = pd.read_csv(fname, sep="\t")

# Display basic information to verify that the file was loaded correctly.
print("\nData preview:")
print(df.head())
print("\nColumn names:")
print(df.columns)
print("\nData types:")
print(df.dtypes)


# ============================================================
# 2. EXPLORATORY DERIVATIVE ANALYSIS
# ============================================================

# Extract time and PSI values from the dataframe.
t = df["INDEX"].values
psi = df["PSI"].values

# Calculate the first derivative of PSI with respect to time.
# This estimates how fast the PSI value changes over time.
dpsi_dt = np.gradient(psi, t)

# Plot raw PSI and its first derivative to visually inspect signal behavior.
plt.figure(figsize=(14, 8))

plt.subplot(2, 1, 1)
plt.plot(t, psi)
plt.ylabel("PSI")
plt.title("PSI")

plt.subplot(2, 1, 2)
plt.plot(t, dpsi_dt)
plt.ylabel("dPSI/dt")

plt.tight_layout()
plt.show()


# ============================================================
# 3. PREPARE, SMOOTH, AND DETECT A PRELIMINARY STABLE PERIOD
# ============================================================

# Convert dataframe columns into NumPy arrays.
t = df["INDEX"].to_numpy()
psi = df["PSI"].to_numpy()

# Smooth the PSI signal using a centered rolling median.
# This helps reduce the impact of spikes or artifacts before calculating derivatives.
W_SMOOTH = 31  # 31 samples; if INDEX is approximately 1 Hz, this is approximately 31 seconds.
psi_s = pd.Series(psi).rolling(W_SMOOTH, center=True, min_periods=1).median().to_numpy()

# Calculate the first and second derivatives of the smoothed PSI signal.
d1 = np.gradient(psi_s, t)     # First derivative: dPSI/dt.
d2 = np.gradient(d1, t)        # Second derivative: d²PSI/dt².

# Ignore the first 5 minutes to avoid awake/induction-related instability.
IGNORE_FIRST = 300  # Approximately 5 minutes if sampling is 1 Hz.
start_idx = np.searchsorted(t, t[0] + IGNORE_FIRST)

# Define a PSI threshold used to identify emergence.
END_PSI = 60
end_candidates = np.where(psi_s > END_PSI)[0]
end_idx = end_candidates[0] if len(end_candidates) else len(t)

# Define the segment between the ignored beginning and the estimated emergence period.
seg = slice(start_idx, end_idx)

# Estimate low-dynamics thresholds from the search segment.
# Lower percentiles make the stability condition stricter.
VEL_THRESH = np.nanpercentile(np.abs(d1[seg]), 20)
ACC_THRESH = np.nanpercentile(np.abs(d2[seg]), 20)

print("\nPreliminary  thresholds:")
print(VEL_THRESH, ACC_THRESH)

# Mark points as stable only when both the first and second derivatives are below their thresholds.
stable = (np.abs(d1) < VEL_THRESH) & (np.abs(d2) < ACC_THRESH)

# Search for the first sustained stable block.
MIN_HOLD = 300  # Approximately 5 minutes if sampling is 1 Hz.
i_star = None

for i in range(start_idx, end_idx - MIN_HOLD):
    if stable[i:i + MIN_HOLD].all():
        i_star = i
        break

print("\nPreliminary maintenance start index:")
print(i_star)

# Plot the raw and smoothed PSI signal with the preliminary maintenance start if detected.
plt.figure(figsize=(14, 5))
plt.plot(t, psi, alpha=0.25, label="PSI raw")
plt.plot(t, psi_s, label="PSI smooth")

if i_star is not None:
    plt.axvline(t[i_star], color="red", linestyle="--", label="i_star (maintenance start)")

plt.axvspan(t[0], t[start_idx], alpha=0.1, label="ignored (induction)")
plt.axvspan(t[end_idx], t[-1], alpha=0.1, label="excluded (emergence)")

plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Plot the first derivative and the velocity threshold lines.
plt.figure(figsize=(14, 7))
plt.subplot(1, 1, 1)
plt.plot(t, d1)
plt.axhline(VEL_THRESH, linestyle="--")
plt.axhline(-VEL_THRESH, linestyle="--")
plt.ylabel("dPSI/dt")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# ============================================================
# 4. PATIENT-SPECIFIC SEARCH WINDOW DETECTION
# ============================================================

# Convert the signal to float arrays for derivative-based calculations.
t = df["INDEX"].to_numpy(dtype=float)   # Time in seconds.
psi = df["PSI"].to_numpy(dtype=float)

# Smooth the signal using a centered rolling mean for continuous derivative calculations.
W_SMOOTH = 31
psi_s = pd.Series(psi).rolling(W_SMOOTH, center=True, min_periods=1).mean().to_numpy()

# Calculate the first derivative of the smoothed PSI signal.
d1 = np.gradient(psi_s, t)

# Detect a large derivative spike, interpreted here as the end of induction or an artifact.
spike_thr = np.nanpercentile(np.abs(d1), 99.5)
spike_idx = np.where(np.abs(d1) >= spike_thr)[0]

if len(spike_idx) == 0:
    raise ValueError("No large spike was detected with p99.5 in |d1|. Lower the percentile to 99.")

# Use the first large early spike for this patient, as done in the original notebook.
last_early_spike = spike_idx[0]
MARGIN = 60
start_search = last_early_spike + MARGIN

# Detect the beginning of emergence using a sustained PSI threshold.
END_PSI = 60
END_HOLD = 120  # 2 minutes sustained.

high = psi_s > END_PSI

end_search = len(t) - 1
for i in range(start_search + END_HOLD, len(t) - END_HOLD):
    if high[i:i + END_HOLD].all():
        end_search = i
        break

# Print diagnostic checks for the detected search window.
print("\nPatient-specific search-window diagnostics:")
print("spike_thr |d1|:", spike_thr)
print("last_early_spike time:", t[last_early_spike])
print("start_search time:", t[start_search])
print("end_search time:", t[end_search])
print("search length (s):", t[end_search] - t[start_search])

if end_search <= start_search + 300:
    raise ValueError("The search window is too short. Adjust END_PSI/END_HOLD or review the signal.")


# ============================================================
# 5. PATIENT-SPECIFIC MAINTENANCE START DETECTION
# ============================================================

# Define the search segment between the estimated post-induction point and emergence onset.
seg = slice(start_search, end_search)

# Calculate a low-dynamics velocity threshold inside the search window.
VEL_P = 30
vel_thr = np.nanpercentile(np.abs(d1[seg]), VEL_P)

# Identify time points where the derivative is below the low-dynamics threshold.
stable = (np.abs(d1) < vel_thr)

# Search for the first mostly stable window.
MIN_HOLD = 300  # 5 minutes.
TOL = 0.95      # At least 95% of the window must be stable.

i_start_maint = None
for i in range(start_search, end_search - MIN_HOLD):
    if stable[i:i + MIN_HOLD].mean() >= TOL:
        i_start_maint = i
        break

print("\nPatient-specific maintenance-start diagnostics:")
print("vel_thr:", vel_thr)
print("mean stable in seg:", stable[seg].mean())
print("i_start_maint:", i_start_maint, "time:", (t[i_start_maint] if i_start_maint is not None else None))

# Plot the patient-specific maintenance search window and detected maintenance start.
plt.figure(figsize=(14, 5))
plt.plot(t, psi, alpha=0.25, label="PSI raw")
plt.plot(t, psi_s, label="PSI smooth")

plt.axvline(t[last_early_spike], color="black", linestyle=":", label="spike (induction/artifact)")
plt.axvline(t[start_search], color="gray", linestyle="--", label="start_search")
plt.axvline(t[end_search], color="gray", linestyle="--", label="end_search (emergence start)")

if i_start_maint is not None:
    plt.axvline(t[i_start_maint], color="red", linestyle="--", label="maintenance start")

plt.axvspan(t[start_search], t[end_search], alpha=0.08, label="search window")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# ============================================================
# 6. ADAPTIVE MAINTENANCE START, END, AND BASELINE FUNCTION
# ============================================================

# Helper function to safely clip an integer value between lower and upper bounds.
def _clip_int(x, lo, hi):
    return int(np.clip(int(x), int(lo), int(hi)))


def maintenance_start_end_and_baseline_adaptive(
    df,
    time_col="INDEX",
    psi_col="PSI",
    # Clinical-ish thresholds (you may "freeze" these later)
    end_psi=60,                 # Emergence threshold.
    # Derivative/refinement settings (mostly robust defaults)
    vel_percentile=50,          # "Low dynamics" threshold percentile inside search window.
    tol=0.90,                   # Allow 10% violations in stability window.
    spike_percentile=99.5,      # Dynamic-event threshold from early reference.
    # Smoothing bounds in seconds; adaptive inside these limits.
    smooth_min=21,
    smooth_max=61,
    # Safety: minimum duration in seconds.
    min_total_duration=600
):
    """
    Detect maintenance start, emergence start, and baseline PSI using an adaptive derivative-based approach.

    Returns a dictionary with:
      - maint_start_time
      - end_search_time
      - baseline
      - arrays for plotting, including psi_smooth and d1

    Assumption:
      - INDEX is in seconds and approximately 1 Hz.
      - INDEX should be monotonic; if it is not, the function sorts the signal by time.
    """

    # --- Load arrays ---
    # Extract time and PSI values from the dataframe.
    t = df[time_col].to_numpy(dtype=float)
    psi = df[psi_col].to_numpy(dtype=float)
    n = len(t)

    # --- Basic checks ---
    # Stop if the signal is too short to support the minimum duration requirement.
    if n < min_total_duration:
        return {"status": "too_short", "reason": f"n={n} < {min_total_duration}s"}

    # Ensure time is monotonic; if not, sort both time and PSI arrays by time.
    if np.any(np.diff(t) < 0):
        order = np.argsort(t)
        t = t[order]
        psi = psi[order]

    # --- Adaptive time parameters, all derived from n seconds ---
    # Smoothing: approximately 0.5% of duration, bounded by smooth_min and smooth_max.
    smooth_win = int(np.round(0.005 * n))
    smooth_win = max(smooth_min, min(smooth_win, smooth_max))
    if smooth_win % 2 == 0:
        smooth_win += 1  # Prefer an odd window for symmetry.

    # Ignore awake/early induction: 5% of case, capped at 5 minutes, with a minimum of 1 minute.
    ignore_awake = int(min(300, 0.05 * n))
    ignore_awake = max(ignore_awake, 60)

    # Tail window used to search for emergence onset: 30% of case, capped at 40 minutes, minimum 5 minutes.
    tail_window = int(min(2400, 0.30 * n))
    tail_window = max(tail_window, 300)

    # Duration required for sustained high PSI during emergence: 5% of case, capped at 2 minutes, minimum 30 seconds.
    end_hold = int(min(120, 0.05 * n))
    end_hold = max(end_hold, 30)

    # Early window used to search induction dynamics: 40% of case, capped at 50 minutes, minimum ignore + 2 minutes.
    max_early = int(min(3000, 0.40 * n))
    max_early = max(max_early, ignore_awake + 120)

    # Reference window used to estimate the spike threshold.
    ref_time = int(min(600, 0.10 * n))
    ref_time = max(ref_time, 120)

    # Stability hold used to define maintenance start: 10% of case, capped at 5 minutes, minimum 2 minutes.
    min_hold = int(min(300, 0.10 * n))
    min_hold = max(min_hold, 120)

    # Baseline window: 5% of case, capped at 60 seconds, minimum 30 seconds.
    baseline_win = int(min(60, 0.05 * n))
    baseline_win = max(baseline_win, 30)

    # Margin after the last early spike: 2% of case, capped at 2 minutes, minimum 30 seconds.
    start_margin = int(min(120, 0.02 * n))
    start_margin = max(start_margin, 30)

    # --- Smooth signal ---
    # Use a centered rolling mean to reduce noise and avoid quantization before derivative calculation.
    psi_s = (
        pd.Series(psi)
        .rolling(smooth_win, center=True, min_periods=1)
        .mean()
        .to_numpy()
    )

    # --- First derivative ---
    # Calculate dPSI/dt from the smoothed signal.
    d1 = np.gradient(psi_s, t)

    # =========================
    # 1) Find emergence onset (end_search) in the tail
    # =========================
    # Emergence is defined as the first sustained block where PSI is above or equal to end_psi.
    tail_start = max(0, n - tail_window)
    high = psi_s >= end_psi

    end_search = None
    for i in range(tail_start, n - end_hold):
        if high[i:i + end_hold].all():
            end_search = i
            break

    # If no sustained emergence block is found, return a diagnostic status.
    if end_search is None:
        return {
            "status": "no_emergence",
            "reason": f"no sustained PSI>={end_psi} in last {tail_window}s",
            "params": {
                "smooth_win": smooth_win,
                "ignore_awake": ignore_awake,
                "tail_window": tail_window,
                "end_hold": end_hold,
            },
        }

    # =========================
    # 2) Find start_search using early dynamic spikes after the ignored awake period
    # =========================
    early_end = min(end_search, max_early)
    if early_end <= ignore_awake + 10:
        # If the case is too short before emergence, use a conservative fallback.
        start_search = ignore_awake
        spike_thr = float("nan")
        spikes = np.array([], dtype=int)
    else:
        # Define the reference range used to estimate the spike threshold.
        ref_end = min(ignore_awake + ref_time, early_end)
        ref_end = max(ref_end, ignore_awake + 10)

        # Calculate a high-percentile derivative threshold for detecting dynamic events.
        spike_thr = np.nanpercentile(np.abs(d1[ignore_awake:ref_end]), spike_percentile)

        # Find derivative spikes in the early part of the case.
        spikes = np.where(np.abs(d1[ignore_awake:early_end]) >= spike_thr)[0] + ignore_awake

        if len(spikes) > 0:
            # Start searching after the last early spike plus a safety margin.
            last_spike = int(spikes.max())
            start_search = last_spike + start_margin
        else:
            # Fallback: start after a bounded post-awake delay.
            start_search = ignore_awake + min(600, max(120, int(0.10 * n)))

    # Clip start_search to valid array bounds.
    start_search = _clip_int(start_search, 0, n - 1)

    # Ensure that start_search and end_search define a valid window.
    if end_search <= start_search + max(min_hold, baseline_win):
        # If the early spike was too late or emergence too early, use ignore_awake as a conservative fallback.
        start_search = _clip_int(ignore_awake, 0, n - 1)

    if end_search <= start_search + max(min_hold, baseline_win):
        return {
            "status": "invalid_window",
            "reason": f"end_search too close to start_search (start={start_search}, end={end_search})",
            "params": {
                "smooth_win": smooth_win,
                "ignore_awake": ignore_awake,
                "tail_window": tail_window,
                "end_hold": end_hold,
                "max_early": max_early,
                "ref_time": ref_time,
                "min_hold": min_hold,
                "baseline_win": baseline_win,
                "start_margin": start_margin,
            },
        }

    # =========================
    # 3) Refine maintenance start using low-d1 stability in [start_search, end_search]
    # =========================
    # Estimate a low-velocity threshold from the search window.
    seg = slice(start_search, end_search)
    vel_thr = np.nanpercentile(np.abs(d1[seg]), vel_percentile)

    # Mark stable points where the absolute first derivative is below the threshold.
    stable = np.abs(d1) < vel_thr

    # Search for the first window that is stable enough according to tol.
    maint_start = None
    for i in range(start_search, end_search - min_hold):
        if stable[i:i + min_hold].mean() >= tol:
            maint_start = i
            break

    # If no stable block is detected, use start_search as a fallback.
    if maint_start is None:
        maint_start = start_search

    # =========================
    # 4) Baseline calculation
    # =========================
    # Calculate baseline as the median smoothed PSI value in the baseline window after maintenance start.
    bw_end = min(maint_start + baseline_win, n)
    baseline = float(np.nanmedian(psi_s[maint_start:bw_end]))

    # Return outputs, arrays, and diagnostics for plotting and debugging.
    return {
        "status": "ok",
        "t": t,
        "psi_raw": psi,
        "psi_smooth": psi_s,
        "d1": d1,
        "start_search_idx": int(start_search),
        "start_search_time": float(t[start_search]),
        "end_search_idx": int(end_search),
        "end_search_time": float(t[end_search]),
        "maint_start_idx": int(maint_start),
        "maint_start_time": float(t[maint_start]),
        "baseline": baseline,
        "diagnostics": {
            "n_seconds": n,
            "smooth_win": smooth_win,
            "ignore_awake": ignore_awake,
            "tail_window": tail_window,
            "end_hold": end_hold,
            "max_early": max_early,
            "ref_time": ref_time,
            "start_margin": start_margin,
            "min_hold": min_hold,
            "baseline_win": baseline_win,
            "spike_thr": float(spike_thr) if "spike_thr" in locals() else float("nan"),
            "num_spikes": int(len(spikes)) if "spikes" in locals() else 0,
            "vel_thr": float(vel_thr),
        },
    }


def plot_maintenance_detection(out, title_extra=""):
    """
    Plot raw PSI, smoothed PSI, maintenance search window, maintenance start, and emergence onset.
    """

    # Do not plot if the detection function did not return a valid result.
    if out.get("status") != "ok":
        print("Cannot plot, status:", out.get("status"), "| reason:", out.get("reason"))
        if "params" in out:
            print("Params:", out["params"])
        return

    # Extract arrays and detected times from the output dictionary.
    t = out["t"]
    psi = out["psi_raw"]
    psi_s = out["psi_smooth"]

    # Plot raw and smoothed PSI signals.
    plt.figure(figsize=(14, 5))
    plt.plot(t, psi, alpha=0.2, label="PSI raw")
    plt.plot(t, psi_s, label="PSI smooth")

    # Add vertical lines for the detected search start, maintenance start, and emergence onset.
    plt.axvline(out["start_search_time"], color="gray", linestyle="--", label="start_search")
    plt.axvline(out["maint_start_time"], color="red", linestyle="--", label="maint_start (d1-stable)")
    plt.axvline(out["end_search_time"], color="gray", linestyle="--", label="end_search (emergence onset)")

    # Shade the detected maintenance window.
    plt.axvspan(out["start_search_time"], out["end_search_time"], alpha=0.08, label="maintenance window")
    plt.title(f'Baseline={out["baseline"]:.2f}  {title_extra}'.strip())
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# ============================================================
# 7. RUN THE ADAPTIVE FUNCTION AND PLOT FINAL RESULTS
# ============================================================

# Run the adaptive maintenance and baseline detection function.
out = maintenance_start_end_and_baseline_adaptive(df)

# Print the final outputs and diagnostics.
print("\nAdaptive detection output:")
print("STATUS:", out["status"])
if out["status"] == "ok":
    print("start_search:", out["start_search_time"])
    print("maint_start :", out["maint_start_time"])
    print("end_search  :", out["end_search_time"])
    print("baseline    :", out["baseline"])
    print("diagnostics :", out["diagnostics"])
    plot_maintenance_detection(out)
else:
    print("reason:", out.get("reason"))
    print("params:", out.get("params"))